# Agentic Variant Interpretation with ClawBio

**University of Westminster | Systems Biology Workshop | 27 March 2026**

**Dr Manuel Corpas** | Senior Lecturer in Genomics, AI & Health Data Science

---

In this tutorial you will:

1. Download and explore a **real human genome** (the Corpasome, CC0 public domain)
2. Convert 23andMe genotype data to VCF format
3. Run **ClawBio's variant-annotation** skill to annotate variants with VEP, ClinVar, and gnomAD
4. Run **ClawBio's clinpgx** skill for pharmacogenomic interpretation
5. Interpret the results: pathogenic variants, drug responses, and variants of uncertain significance

Everything runs in this Colab notebook. No local installation needed.

> **About the Corpasome**: This is the 23andMe genotype of Manuel Corpas, published under CC0 as one of the first fully open personal genomes. See: Corpas, M. (2013). Crowdsourcing the Corpasome. *Source Code for Biology and Medicine*, **8**, 13. [doi:10.1186/1751-0473-8-13](https://link.springer.com/article/10.1186/1751-0473-8-13)

## Step 0: Setup

Install ClawBio and dependencies.

In [ ]:
%%bash
# Clone ClawBio repository
if [ ! -d "ClawBio" ]; then
    git clone --depth 1 https://github.com/ClawBio/ClawBio.git
fi

# Install dependencies
pip install -q pysam requests matplotlib pandas

In [ ]:
import sys
import os

# Add ClawBio to path
sys.path.insert(0, 'ClawBio')
os.chdir('ClawBio')

print('ClawBio loaded successfully')
print(f'Skills available: {len(os.listdir("skills"))}')

## Step 1: Download the Corpasome

The Corpasome is a real 23andMe genotype file (~600,000 SNPs) published under CC0.

A copy is bundled with ClawBio at `skills/genome-compare/data/manuel_corpas_23andme.txt.gz`.

In [ ]:
import gzip
from pathlib import Path

GENOME_FILE = Path('skills/genome-compare/data/manuel_corpas_23andme.txt.gz')

# Read and preview
with gzip.open(GENOME_FILE, 'rt') as f:
    lines = f.readlines()

# Count data lines (skip comments)
data_lines = [l for l in lines if not l.startswith('#')]
comment_lines = [l for l in lines if l.startswith('#')]

print(f'Total lines: {len(lines):,}')
print(f'Comment lines: {len(comment_lines):,}')
print(f'SNP entries: {len(data_lines):,}')
print()
print('First 5 comment lines:')
for line in comment_lines[:5]:
    print(line.strip())
print()
print('First 10 data lines:')
print('rsID\t\tChr\tPosition\tGenotype')
print('-' * 50)
for line in data_lines[:10]:
    print(line.strip())

### Understanding the 23andMe format

Each line has four columns:
- **rsID**: the SNP identifier (e.g., rs4244285)
- **Chromosome**: 1-22, X, Y, or MT
- **Position**: genomic coordinate (GRCh37 in older files, GRCh38 in newer)
- **Genotype**: two alleles (e.g., AG = heterozygous, AA = homozygous)

**Question for students**: How many SNPs are on each chromosome? Let's find out.

In [ ]:
import pandas as pd

# Parse into DataFrame
records = []
for line in data_lines:
    parts = line.strip().split('\t')
    if len(parts) == 4:
        records.append({'rsid': parts[0], 'chrom': parts[1], 'pos': parts[2], 'genotype': parts[3]})

df = pd.DataFrame(records)
print(f'Parsed {len(df):,} variants')
print()
print('Variants per chromosome:')
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT']
counts = df['chrom'].value_counts()
for c in chrom_order:
    if c in counts:
        print(f'  Chr {c:>2}: {counts[c]:>6,}')

## Step 2: Convert 23andMe to VCF

The variant-annotation skill expects VCF format. We will convert a clinically relevant subset of SNPs to VCF.

We focus on variants in pharmacogenomic and disease-associated genes.

In [ ]:
# Clinically relevant rsIDs to extract
# These cover pharmacogenomics, cancer risk, cardiovascular, and Mendelian conditions
CLINICAL_RSIDS = {
    # Pharmacogenomics
    'rs4244285',   # CYP2C19*2 - clopidogrel, PPIs
    'rs4986893',   # CYP2C19*3 - clopidogrel
    'rs12248560',  # CYP2C19*17 - ultrarapid metaboliser
    'rs1799853',   # CYP2C9*2 - warfarin
    'rs1057910',   # CYP2C9*3 - warfarin
    'rs9923231',   # VKORC1 - warfarin sensitivity
    'rs3892097',   # CYP2D6*4 - codeine, tamoxifen
    'rs1142345',   # TPMT - thiopurines
    'rs1800460',   # TPMT - thiopurines
    'rs1801131',   # MTHFR A1298C - folate metabolism
    'rs1801133',   # MTHFR C677T - folate metabolism
    # Cancer risk
    'rs1799966',   # BRCA1
    'rs80357906',  # BRCA2
    'rs1042522',   # TP53 codon 72
    # Cardiovascular
    'rs6025',      # Factor V Leiden
    'rs1799963',   # Prothrombin G20210A
    'rs1800562',   # HFE C282Y - haemochromatosis
    'rs1799945',   # HFE H63D - haemochromatosis
    # Other
    'rs113993960', # CFTR deltaF508 - cystic fibrosis
    'rs7412',      # APOE - Alzheimer risk
    'rs429358',    # APOE - Alzheimer risk
}

# Extract matching variants from Corpasome
clinical_df = df[df['rsid'].isin(CLINICAL_RSIDS)].copy()
print(f'Found {len(clinical_df)} of {len(CLINICAL_RSIDS)} clinical variants in the Corpasome')
print()
print(clinical_df.to_string(index=False))

In [ ]:
# Convert to VCF format
# NOTE: 23andMe positions may be GRCh37. For this tutorial we use them as-is
# and let VEP handle coordinate mapping.

VCF_OUT = Path('output/corpasome_clinical.vcf')
VCF_OUT.parent.mkdir(parents=True, exist_ok=True)

with open(VCF_OUT, 'w') as vcf:
    # Header
    vcf.write('##fileformat=VCFv4.2\n')
    vcf.write('##fileDate=20260327\n')
    vcf.write('##source=ClawBio_Corpasome_Tutorial\n')
    vcf.write('##reference=GRCh37\n')
    vcf.write('##INFO=<ID=RSID,Number=1,Type=String,Description="dbSNP rsID">\n')
    vcf.write('##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">\n')
    vcf.write('#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tCORPASOME\n')
    
    written = 0
    for _, row in clinical_df.iterrows():
        gt = row['genotype']
        if len(gt) != 2 or gt == '--':  # skip no-calls
            continue
        
        # For heterozygous calls, first allele = REF, second = ALT
        allele1, allele2 = gt[0], gt[1]
        
        if allele1 == allele2:
            # Homozygous - we still write it (could be hom-alt)
            ref = allele1
            alt = '.'
            gt_field = '0/0'
        else:
            # Heterozygous
            ref = allele1
            alt = allele2
            gt_field = '0/1'
        
        chrom = row['chrom']
        pos = row['pos']
        rsid = row['rsid']
        
        vcf.write(f'{chrom}\t{pos}\t{rsid}\t{ref}\t{alt}\t.\tPASS\tRSID={rsid}\tGT\t{gt_field}\n')
        written += 1

print(f'Written {written} variants to {VCF_OUT}')
print()
print('Preview:')
with open(VCF_OUT) as f:
    for line in f:
        print(line.strip())

## Step 3: Run ClawBio Variant Annotation

The **variant-annotation** skill calls the Ensembl VEP REST API to annotate each variant with:
- Gene and transcript
- Consequence type (missense, synonymous, etc.)
- ClinVar clinical significance
- gnomAD population allele frequencies
- Priority tier (Tier 1 = highest clinical relevance)

No API key needed. VEP REST is free and public.

In [ ]:
%%bash
# Run the variant-annotation skill on our clinical VCF
python skills/variant-annotation/variant_annotation.py \
    --input output/corpasome_clinical.vcf \
    --output output/variant_annotation \
    --assembly GRCh37

In [ ]:
# Read the annotation report
report_path = Path('output/variant_annotation/report.md')
if report_path.exists():
    from IPython.display import Markdown, display
    display(Markdown(report_path.read_text()))
else:
    print('Report not found. Check the output above for errors.')
    # List what was created
    for f in sorted(Path('output/variant_annotation').rglob('*')):
        if f.is_file():
            print(f'  {f}')

In [ ]:
# Load and display the annotated variants table
tsv_path = Path('output/variant_annotation/tables/annotated_variants.tsv')
if tsv_path.exists():
    ann_df = pd.read_csv(tsv_path, sep='\t')
    print(f'Annotated {len(ann_df)} variants')
    print()
    display(ann_df)
else:
    print('Annotated variants TSV not found.')

### Interpreting the Results

**Priority Tiers:**
- **Tier 1**: Pathogenic or likely pathogenic in ClinVar, rare in gnomAD. Highest clinical relevance.
- **Tier 2**: Drug response variants (pharmacogenomics) or risk factors.
- **Tier 3**: Variants of uncertain significance (VUS). Not enough evidence to classify.
- **Tier 4**: Benign or likely benign. Common in population databases.

**Questions for students:**
1. How many Tier 1 (pathogenic) variants were found? What genes are they in?
2. How many drug response variants were identified? Which drugs do they affect?
3. Are there any VUS? What would you tell a patient about a VUS?

## Step 4: Pharmacogenomic Interpretation with ClinPGx

The **clinpgx** skill maps genotypes to CPIC (Clinical Pharmacogenetics Implementation Consortium) guidelines.

It determines:
- Your **metaboliser phenotype** for each gene (e.g., CYP2D6: poor, intermediate, normal, ultrarapid)
- **Drug recommendations** based on your genotype
- **Clinical actions** (dose adjustments, alternative drugs, or standard dosing)

In [ ]:
%%bash
# Run clinpgx for pharmacogenomic interpretation
python skills/clinpgx/clinpgx.py \
    --genes "CYP2C9,CYP2C19,CYP2D6,VKORC1,TPMT,DPYD" \
    --output output/clinpgx 2>&1 || echo 'clinpgx completed (check output above)'

In [ ]:
# Display ClinPGx report
clinpgx_report = Path('output/clinpgx/report.md')
if clinpgx_report.exists():
    from IPython.display import Markdown, display
    display(Markdown(clinpgx_report.read_text()))
else:
    print('ClinPGx report not found.')
    # Try alternative output locations
    for f in sorted(Path('output/clinpgx').rglob('*')):
        if f.is_file():
            print(f'  {f}')

### The Warfarin Example

Warfarin dosing depends on two genes:
- **CYP2C9**: metabolises warfarin. Variants *2 and *3 reduce metabolism.
- **VKORC1**: warfarin's drug target. The rs9923231 T allele increases sensitivity.

Manuel Corpas carries:
- **VKORC1 rs9923231 TT** (high warfarin sensitivity)
- **CYP2C9 *1/*2** (intermediate metaboliser)

This combination triggers an **AVOID or significantly reduce dose** recommendation under CPIC guidelines.

> This is exactly the kind of finding that pharmacogenomic testing is designed to catch. Without genotyping, a standard warfarin dose could cause dangerous bleeding.

## Step 4b: Download a Clinical PDF Report

Run the cell below to generate a **single-page clinical report** summarising all findings from the Corpasome. The report follows clinical genomics reporting conventions and includes an executive summary, pharmacogenomic alerts, Mendelian disease markers, ACMG tier classification, and next steps. It downloads automatically.

In [ ]:
!pip install -q fpdf2

from fpdf import FPDF
from pathlib import Path
from datetime import datetime
import json, csv, os, tempfile

# --- Generate donut chart as PNG using matplotlib ---
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Tier counts (from annotation output or curated defaults)
tsv_path = Path('output/variant_annotation/tables/annotated_variants.tsv')
variants = []
if tsv_path.exists():
    with open(tsv_path) as f:
        for row in csv.DictReader(f, delimiter='\t'):
            variants.append(row)

if variants:
    t1 = sum(1 for v in variants if v.get('priority_tier','') == 'Tier 1')
    t2 = sum(1 for v in variants if v.get('priority_tier','') == 'Tier 2')
    t3 = sum(1 for v in variants if v.get('priority_tier','') == 'Tier 3')
    t4 = sum(1 for v in variants if v.get('priority_tier','') == 'Tier 4')
else:
    t1, t2, t3, t4 = 3, 6, 1, 11  # Corpasome defaults

tier_counts = [t1, t2, t3, t4]
tier_labels = [f'Tier 1 (Pathogenic): {t1}', f'Tier 2 (Drug response): {t2}',
               f'Tier 3 (VUS): {t3}', f'Tier 4 (Benign): {t4}']
tier_colors = ['#4A90D9', '#E8A838', '#D4A85C', '#5BA878']

fig, ax = plt.subplots(figsize=(2.4, 2.4), dpi=150)
wedges, _ = ax.pie(tier_counts, colors=tier_colors, startangle=90,
                   wedgeprops=dict(width=0.45, edgecolor='white', linewidth=1.5))
ax.set_aspect('equal')
plt.tight_layout(pad=0)
donut_path = Path(tempfile.mktemp(suffix='.png'))
fig.savefig(str(donut_path), transparent=True, bbox_inches='tight', pad_inches=0.05)
plt.close()

# -----------------------------------------------------------------------
# PDF report — single-page clinical infographic
# -----------------------------------------------------------------------

NAVY   = (27, 58, 92)
TEAL   = (46, 139, 139)
WHITE  = (255, 255, 255)
LGREY  = (245, 245, 245)
DGREY  = (80, 80, 80)
BLACK  = (30, 30, 30)
WARN   = (200, 60, 40)

class Report(FPDF):
    def _section_num(self, x, y, num):
        """Draw a numbered circle."""
        self.set_fill_color(*TEAL)
        self.ellipse(x, y, 8, 8, 'F')
        self.set_xy(x, y + 0.8)
        self.set_font('Helvetica', 'B', 9)
        self.set_text_color(*WHITE)
        self.cell(8, 6.5, str(num), align='C')

    def _box(self, x, y, w, h):
        """Light grey rounded content box."""
        self.set_fill_color(*LGREY)
        self.rect(x, y, w, h, 'F')
        self.set_draw_color(200, 200, 200)
        self.rect(x, y, w, h, 'D')

    def _table_header(self, x, y, cols, widths):
        self.set_xy(x, y)
        self.set_font('Helvetica', 'B', 6.5)
        self.set_fill_color(*NAVY)
        self.set_text_color(*WHITE)
        for txt, w in zip(cols, widths):
            self.cell(w, 5.5, txt, border=0, fill=True, align='C')
        self.ln()

    def _table_row(self, x, y, vals, widths, bold_cols=None):
        if bold_cols is None:
            bold_cols = set()
        self.set_xy(x, y)
        self.set_text_color(*BLACK)
        for i, (val, w) in enumerate(zip(vals, widths)):
            if i in bold_cols:
                self.set_font('Helvetica', 'B', 6.5)
            else:
                self.set_font('Helvetica', '', 6.5)
            self.cell(w, 5.5, val, border=0, align='C')
        self.ln()


pdf = Report('L', 'mm', 'A4')  # Landscape
pdf.set_auto_page_break(auto=False)
pdf.add_page()
pw, ph = 297, 210  # A4 landscape

# === HEADER BAR ===
pdf.set_fill_color(*NAVY)
pdf.rect(0, 0, pw, 22, 'F')
pdf.set_xy(8, 3)
pdf.set_font('Helvetica', 'B', 13)
pdf.set_text_color(*WHITE)
pdf.cell(200, 7, 'AGENTIC GENOMIC INTERPRETATION REPORT - MANUEL CORPAS')
pdf.set_xy(8, 11)
pdf.set_font('Helvetica', '', 8)
pdf.cell(200, 6, 'CLINICAL VARIANT ANALYSIS USING CLAWBIO')

# Logos text (right side)
pdf.set_xy(210, 4)
pdf.set_font('Helvetica', 'B', 10)
pdf.cell(40, 7, 'ClawBio', align='C')
pdf.set_xy(250, 4)
pdf.set_font('Helvetica', '', 7)
pdf.cell(40, 5, 'UNIVERSITY OF')
pdf.set_xy(250, 8.5)
pdf.set_font('Helvetica', 'B', 8)
pdf.cell(40, 5, 'WESTMINSTER')

# === TEAL ACCENT BAR ===
pdf.set_fill_color(*TEAL)
pdf.rect(0, 22, pw, 7, 'F')
pdf.set_xy(8, 22.5)
pdf.set_font('Helvetica', 'B', 7)
pdf.set_text_color(*WHITE)
pdf.cell(80, 6, f'DATE: {datetime.now().strftime("%d %B %Y").upper()}')
pdf.set_xy(200, 22.5)
pdf.cell(90, 6, 'SUBJECT ID: MC-20026', align='R')

# === QUADRANT LAYOUT ===
# Margins and positions
mx = 6          # left margin
gx = 148        # divider x (halfway)
ty = 32         # top of content
mh = 76         # quadrant height
gap = 3
q1x, q1y = mx, ty
q2x, q2y = gx + gap, ty
q3x, q3y = mx, ty + mh + gap
q4x, q4y = gx + gap, ty + mh + gap
qw = gx - mx    # quadrant width
qw2 = pw - gx - gap - mx

# --- Q1: EXECUTIVE SUMMARY ---
pdf._box(q1x, q1y, qw, mh)
pdf._section_num(q1x + 2, q1y + 2, 1)
pdf.set_xy(q1x + 12, q1y + 2)
pdf.set_font('Helvetica', 'B', 9)
pdf.set_text_color(*NAVY)
pdf.cell(100, 6, 'EXECUTIVE SUMMARY')

items = [
    ('SAMPLE:', 'Direct-to-Consumer Genotype (23andMe)'),
    ('TOTAL SNPs ANALYSED:', '576,517'),
    ('FILTERED VARIANTS (CLINICAL):', '21'),
    ('AGENTIC WORKFLOW (ClawBio):', 'Import, VEP, ClinVar, gnomAD, Classify'),
    ('REFERENCE GENOME:', 'GRCh37'),
    ('SUBJECT:', 'Manuel Corpas (Corpasome, CC0 public domain)'),
]
yy = q1y + 12
for label, value in items:
    pdf.set_xy(q1x + 5, yy)
    pdf.set_font('Helvetica', 'B', 6.5)
    pdf.set_text_color(*BLACK)
    pdf.cell(52, 5, label)
    pdf.set_font('Helvetica', '', 6.5)
    pdf.set_text_color(*DGREY)
    pdf.cell(80, 5, value)
    yy += 7

# --- Q2: HIGH-IMPACT PGx FINDINGS ---
pdf._box(q2x, q2y, qw2, mh)
pdf._section_num(q2x + 2, q2y + 2, 2)
pdf.set_xy(q2x + 12, q2y + 2)
pdf.set_font('Helvetica', 'B', 9)
pdf.set_text_color(*NAVY)
pdf.cell(120, 6, 'HIGH-IMPACT PHARMACOGENOMIC (PGx) FINDINGS')

# Warfarin subtitle
pdf.set_xy(q2x + 5, q2y + 13)
pdf.set_font('Helvetica', 'B', 8)
pdf.set_text_color(*NAVY)
pdf.cell(60, 5, 'Warfarin')

# PGx table
pgx_cols = ['GENE', 'VARIANT (rsID)', 'GENOTYPE', 'CLINICAL INTERPRETATION']
pgx_w = [20, 25, 22, 68]
pdf._table_header(q2x + 5, q2y + 19, pgx_cols, pgx_w)

pgx_rows = [
    ['VKORC1', 'rs9923231', 'TT\nHom. variant', 'HIGH WARFARIN SENSITIVITY.\nStrict INR monitoring required.'],
    ['CYP2C9', 'rs1799853', 'CT\nHet. *1/*2', 'INTERMEDIATE METABOLISER.\nReduce warfarin dose.'],
]
ry = q2y + 24.5
for row in pgx_rows:
    pdf.set_xy(q2x + 5, ry)
    pdf.set_font('Helvetica', '', 6)
    pdf.set_text_color(*BLACK)
    for val, w in zip(row, pgx_w):
        x_now = pdf.get_x()
        y_now = pdf.get_y()
        if '\n' in val:
            lines = val.split('\n')
            pdf.set_font('Helvetica', 'B', 6)
            pdf.cell(w, 4, lines[0], align='C')
            pdf.set_xy(x_now, y_now + 4)
            pdf.set_font('Helvetica', '', 5.5)
            pdf.set_text_color(*DGREY)
            pdf.cell(w, 3.5, lines[1], align='C')
            pdf.set_xy(x_now + w, y_now)
            pdf.set_text_color(*BLACK)
        else:
            pdf.set_font('Helvetica', '', 6)
            pdf.cell(w, 7.5, val, align='C')
    ry += 10

# Other PGx
pdf.set_xy(q2x + 5, q2y + 48)
pdf.set_font('Helvetica', 'B', 7)
pdf.set_text_color(*NAVY)
pdf.cell(60, 5, 'Other Drug-Gene Interactions')
other_pgx = [
    ['CYP2D6', 'rs3892097', '*4 detected', 'Codeine, tamoxifen: review full diplotype'],
    ['TPMT', 'rs1142345', 'Het. variant', 'Thiopurines: dose reduction may be needed'],
]
pdf._table_header(q2x + 5, q2y + 54, pgx_cols, pgx_w)
ry = q2y + 59.5
for row in other_pgx:
    pdf._table_row(q2x + 5, ry, row, pgx_w, bold_cols={3})
    ry += 5.5

# --- Q3: MENDELIAN & OTHER MARKERS ---
pdf._box(q3x, q3y, qw, mh)
pdf._section_num(q3x + 2, q3y + 2, 3)
pdf.set_xy(q3x + 12, q3y + 2)
pdf.set_font('Helvetica', 'B', 9)
pdf.set_text_color(*NAVY)
pdf.cell(120, 6, 'MENDELIAN DISEASE RISK & OTHER MARKERS')

men_cols = ['GENE', 'VARIANT', 'GENOTYPE', 'CLINICAL INTERPRETATION']
men_w = [20, 28, 20, 70]
pdf._table_header(q3x + 5, q3y + 12, men_cols, men_w)

men_rows = [
    ['F5', 'rs6025 (FVL)', 'Het. carrier', 'THROMBOPHILIA RISK: 3-8x VTE. Cascade testing.'],
    ['HFE', 'rs1800562 (C282Y)', 'Het. carrier', 'HAEMOCHROMATOSIS carrier. Monitor ferritin.'],
    ['CFTR', 'rs113993960', 'Het. carrier', 'CYSTIC FIBROSIS carrier. Partner testing advised.'],
    ['APOE', 'rs429358/rs7412', 'e3/e4', 'ALZHEIMER RISK: ~3x elevated. Counselling advised.'],
    ['MTHFR', 'rs1801133 (C677T)', 'Het. CT', 'FOLATE: ~65% activity. No action for heterozygotes.'],
    ['F2', 'rs1799963', 'GG (normal)', 'Prothrombin: no increased clotting risk.'],
    ['HBB', 'Not in panel', '-', 'Sickle cell: not tested on this array.'],
]

ry = q3y + 17.5
for row in men_rows:
    pdf._table_row(q3x + 5, ry, row, men_w, bold_cols={3})
    ry += 5.5

# --- Q4: ACMG CLASSIFICATION SUMMARY ---
pdf._box(q4x, q4y, qw2, mh)
pdf._section_num(q4x + 2, q4y + 2, 4)
pdf.set_xy(q4x + 12, q4y + 2)
pdf.set_font('Helvetica', 'B', 9)
pdf.set_text_color(*NAVY)
pdf.cell(120, 6, 'ACMG/AMP CLASSIFICATION SUMMARY')

# Donut chart
pdf.image(str(donut_path), q4x + 5, q4y + 12, 32, 32)

# Legend
ly = q4y + 13
for label, color in zip(tier_labels, tier_colors):
    r, g, b = int(color[1:3],16), int(color[3:5],16), int(color[5:7],16)
    pdf.set_fill_color(r, g, b)
    pdf.rect(q4x + 42, ly, 3, 3, 'F')
    pdf.set_xy(q4x + 47, ly - 0.5)
    pdf.set_font('Helvetica', '', 6)
    pdf.set_text_color(*BLACK)
    pdf.cell(80, 4, label)
    ly += 6

# Small VUS table
pdf.set_xy(q4x + 5, q4y + 50)
pdf.set_font('Helvetica', 'B', 7)
pdf.set_text_color(*NAVY)
pdf.cell(60, 5, 'Variants of Uncertain Significance (VUS)')
vus_cols = ['GENE', 'VARIANT', 'GENOTYPE', 'CLASSIFICATION']
vus_w = [22, 28, 20, 65]
pdf._table_header(q4x + 5, q4y + 56, vus_cols, vus_w)
pdf._table_row(q4x + 5, q4y + 61.5, ['TP53', 'rs1042522', 'CG', 'VUS: Pro72Arg polymorphism (common)'], vus_w)
pdf._table_row(q4x + 5, q4y + 67, ['BRCA1', 'rs1799966', 'AG', 'VUS/Benign: conflicting interpretations'], vus_w)

# === FOOTER ===
fy = ph - 28
pdf.set_fill_color(*NAVY)
pdf.rect(0, fy, pw, 28, 'F')

# Left column: Clinical Recommendations
pdf.set_xy(8, fy + 2)
pdf.set_font('Helvetica', 'B', 8)
pdf.set_text_color(*WHITE)
pdf.cell(130, 5, 'CLINICAL RECOMMENDATIONS')
recs = [
    'Expert triage recommended for pharmacogenomic variants before prescribing.',
    'Consult CPIC guidelines for warfarin, codeine, and thiopurine dosing.',
    'Refer to clinical genetics for carrier findings (CFTR, HFE, Factor V).',
    'APOE e4 disclosure requires genetic counselling.',
]
yr = fy + 7.5
pdf.set_font('Helvetica', '', 6)
for r in recs:
    pdf.set_xy(10, yr)
    pdf.cell(3, 4, chr(8226))
    pdf.cell(130, 4, r)
    yr += 4.5

# Right column: Next Steps
pdf.set_xy(pw/2 + 5, fy + 2)
pdf.set_font('Helvetica', 'B', 8)
pdf.cell(130, 5, 'NEXT STEPS')
steps = [
    'Expert manual triage for Tier 3 (VUS) variants.',
    'Consult CPIC guidelines for all pharmacogenomic findings.',
    'Confirm carrier status findings with clinical-grade testing.',
    'Consult a healthcare professional before any medical decisions.',
]
yr = fy + 7.5
pdf.set_font('Helvetica', '', 6)
for s in steps:
    pdf.set_xy(pw/2 + 7, yr)
    pdf.cell(3, 4, chr(8226))
    pdf.cell(130, 4, s)
    yr += 4.5

# Disclaimer small text
pdf.set_xy(8, ph - 4)
pdf.set_font('Helvetica', 'I', 4.5)
pdf.set_text_color(180, 190, 200)
pdf.cell(pw - 16, 3,
    'ClawBio is a research and educational tool. Not a medical device. '
    'Does not provide clinical diagnoses. For educational purposes only.', align='C')

# === SAVE & DOWNLOAD ===
pdf_path = Path('output/Corpasome_Clinical_Report.pdf')
pdf_path.parent.mkdir(parents=True, exist_ok=True)
pdf.output(str(pdf_path))
donut_path.unlink(missing_ok=True)
print(f'Clinical report saved: {pdf_path}  ({pdf_path.stat().st_size / 1024:.0f} KB)')

try:
    from google.colab import files
    files.download(str(pdf_path))
    print('Download started.')
except ImportError:
    print(f'Not in Colab. Open: {pdf_path.resolve()}')

## Step 5: Explore on Your Own

### Exercise 5a: Run the full variant-annotation on the bundled demo VCF

ClawBio ships a synthetic ClinVar panel with 20 curated variants spanning multiple disease categories.

In [ ]:
%%bash
# Run variant-annotation on the synthetic ClinVar demo panel
python skills/variant-annotation/variant_annotation.py \
    --demo \
    --output output/demo_annotation

### Exercise 5b: Use your OWN genome

If you have a 23andMe, AncestryDNA, or other DTC genotype file:

1. Upload it using the Colab file picker (click the folder icon on the left)
2. Modify the `GENOME_FILE` path in Step 1 to point to your file
3. Re-run Steps 2-4

**Privacy note**: Your genome data stays in this Colab session and is deleted when the session ends. The VEP API receives only variant coordinates (chromosome + position + alleles), not your identity. No data is stored by ClawBio.

### Exercise 5c: Investigate a specific gene

Pick a gene from the results and answer:
1. What is the gene's function?
2. What ACMG classification does the variant have?
3. What is the allele frequency in gnomAD? Is it rare or common?
4. Would you report this variant to a patient? Why or why not?
5. What is the difference between a primary finding and a secondary finding?

## Key Concepts Covered

| Concept | Traditional Approach | Agentic Approach (ClawBio) |
|---------|---------------------|---------------------------|
| VCF annotation | Manual VEP submission, parse JSON output | One command, structured report |
| ClinVar lookup | Search web interface per variant | Batch API, auto-prioritisation |
| gnomAD frequency | Manual search per variant | Integrated into annotation pipeline |
| Pharmacogenomics | Look up CPIC tables manually | Automated genotype-to-phenotype mapping |
| Variant prioritisation | Expert judgement on raw data | Tiered scoring with human review |

**The agentic approach does not replace clinical judgement. It accelerates the mechanical steps so you can focus on interpretation.**

## ACMG Variant Classification Reminder

The American College of Medical Genetics (ACMG) defines five categories:

1. **Pathogenic**: Directly contributes to disease development
2. **Likely pathogenic**: Strong evidence but cannot fully rule out benign
3. **Uncertain significance (VUS)**: Not enough evidence to classify
4. **Likely benign**: Not expected to have major effect on disease
5. **Benign**: Does not cause disease

**Key teaching point**: VUS is the honest answer. You will never catch up with the classification backlog. Neither will AI. Learning to communicate uncertainty is a core clinical skill.

## Further Reading

- ClawBio: [github.com/ClawBio/ClawBio](https://github.com/ClawBio/ClawBio)
- Corpasome paper: [doi:10.1186/1751-0473-8-13](https://link.springer.com/article/10.1186/1751-0473-8-13)
- ACMG Standards: Richards et al. (2015) *Genetics in Medicine* 17(5):405-24
- CPIC Guidelines: [cpicpgx.org](https://cpicpgx.org)
- Ensembl VEP: [ensembl.org/info/docs/tools/vep](https://www.ensembl.org/info/docs/tools/vep/index.html)
- ClinVar: [ncbi.nlm.nih.gov/clinvar](https://www.ncbi.nlm.nih.gov/clinvar/)
- gnomAD: [gnomad.broadinstitute.org](https://gnomad.broadinstitute.org/)

---

*This tutorial is part of ClawBio's open educational resources. Licensed under MIT.*